# Dawah Dialectics V2 (`yt-spx`): Multi-Stage Acoustic Diarization, Overlap Separation & Local LLM Pipeline
### Dynamic Free Colab T4 Sessions (1–8 hrs variable availability) + 5 TB Google Drive + Multi-Account Parallelism

**How this V2 setup works (`yt-spx` — completely separate from legacy `yt-ch` archive):**
1. **Micro-Step Checkpointing (`M1 -> M6`)**: Every single sub-step (`M1` temp audio fetch, `M2` 15-min Demucs vocal chunks + 64kbps `.opus`, `M3` NeMo MSDD diarization + 192-D TitaNet voiceprints, `M4` MossFormer2 2-speaker overlap separation, `M5` Faster-Whisper + CTC word-level scenario JSON, `M6` episode-by-episode Qwen 2.5 14B extraction) saves immediately to `/content/drive/MyDrive/yt-spx`. If Colab disconnects at any second, nothing is redone.
2. **Anti-Ban Smart Batches**: Processes small safe batches (`--batch-videos 6`, `--max-session-minutes 165`), always finishes partially-started videos on Drive *before* touching YouTube, and automatically switches to offline GPU mode if YouTube triggers a rate-limit.
3. **Multi-Account Google Drive Sharing**:
   - On your **5 TB Google One Pro account**, let Cell 1 create `/content/drive/MyDrive/yt-spx` (and optionally upload one Netscape cookies file per Google account into `/content/drive/MyDrive/yt-spx/cookies/` — e.g. `acct1.txt`, `acct2.txt`. The pipeline tests them, quarantines any that get bot-checked / logged out, and automatically rotates to the next one).
   - Right-click the `yt-spx` folder in Google Drive -> **Share** -> grant **Editor** access to your secondary Colab Google accounts.
   - On each secondary Google account, open **Shared with me**, right-click `yt-spx` -> **Organize** -> **Add shortcut** -> place it inside **My Drive**. Every account now sees `/content/drive/MyDrive/yt-spx` and coordinates automatically via atomic lock files!

*(Ensure Runtime is set to **T4 GPU**: `Runtime -> Change runtime type -> T4 GPU`)*

In [ ]:
# ==============================================================================
# CELL 1: 1-CLICK SMART-BATCH AUTO MODE (WORKS ACROSS ANY 1-8 HR COLAB SESSION)
# Runs Stage 1 (M1->M5) in an isolated GPU process, frees 100% VRAM, then runs
# Stage 2 Qwen 2.5 14B (M5->M6) and compacts shards into dawah_knowledge.db.
# Micro-step checkpoints ensure ANY disconnect (after 1h or 8h) loses ZERO progress.
# ==============================================================================
from google.colab import drive
import os, sys

drive.mount('/content/drive')

repo_url = 'https://github.com/ibrahim4433/yt-spx.git'
code_dir = '/content/yt-spx'

if not os.path.exists(code_dir):
    !git clone {repo_url} {code_dir}
else:
    !cd {code_dir} && git pull

os.chdir(code_dir)

# Customize batch size, session timeout, or worker label per Colab account:
WORKER_NAME = 'colab-t4-acct1'
BATCH_VIDEOS = 6
# Set MAX_MINUTES to 0 for unlimited continuous running (runs until Colab disconnects),
# or set a specific cutoff (e.g. 165 for ~2.75h, 300 for 5h, 480 for 8h):
MAX_MINUTES = 0

!PYTHONPATH=/content/yt-spx python run_colab_all_in_one.py \
    --mode auto \
    --batch-videos {BATCH_VIDEOS} \
    --stage2-batch-videos 8 \
    --max-session-minutes {MAX_MINUTES} \
    --debates-only \
    --worker-id {WORKER_NAME}


In [ ]:
# ==============================================================================
# CELL 2 (OPTIONAL): MULTI-ACCOUNT SPECIALIST WORKER MODE
# Set ROLE = 'stage1' on Account A (Audio Cleaning + NeMo Diarization + Overlaps)
# Set ROLE = 'stage2' on Account B (Qwen 2.5 14B Dialectical Extraction)
# Both accounts work simultaneously on the shared 5 TB Drive with zero conflicts!
# ==============================================================================
from google.colab import drive
import os

drive.mount('/content/drive')
repo_url = 'https://github.com/ibrahim4433/yt-spx.git'
code_dir = '/content/yt-spx'
if not os.path.exists(code_dir):
    !git clone {repo_url} {code_dir}
else:
    !cd {code_dir} && git pull
os.chdir(code_dir)

ROLE = 'stage1'          # Change to 'stage2' on your second Colab account
WORKER_NAME = 'colab-specialist-1'

!PYTHONPATH=/content/yt-spx python run_colab_all_in_one.py \
    --mode {ROLE} \
    --batch-videos 8 \
    --stage2-batch-videos 10 \
    --max-session-minutes 170 \
    --debates-only \
    --worker-id {WORKER_NAME}


In [ ]:
# ==============================================================================
# CELL 3: INSTANT 5 TB DRIVE PROGRESS TELEMETRY & SQLITE SHARD COMPACTOR
# Safe to run anytime (even on a CPU runtime) to inspect M0->M6 counts & locks.
# ==============================================================================
from google.colab import drive
import os

drive.mount('/content/drive')
repo_url = 'https://github.com/ibrahim4433/yt-spx.git'
code_dir = '/content/yt-spx'
if not os.path.exists(code_dir):
    !git clone {repo_url} {code_dir}
os.chdir(code_dir)

!PYTHONPATH=/content/yt-spx python run_colab_all_in_one.py --mode sync-db


In [ ]:
# ==============================================================================
# CELL 4 (OPTIONAL): TEST EVERY COOKIES FILE IN /content/drive/MyDrive/yt-spx/cookies/
# Runs one yt-dlp --simulate request per cookie, marks dead ones as quarantined in the
# shared registry (locks/_cookie_health.json). Replacing a file clears its quarantine.
# ==============================================================================
from google.colab import drive
import os

drive.mount('/content/drive')
repo_url = 'https://github.com/ibrahim4433/yt-spx.git'
code_dir = '/content/yt-spx'
if not os.path.exists(code_dir):
    !git clone {repo_url} {code_dir}
else:
    !cd {code_dir} && git pull
os.chdir(code_dir)

!PYTHONPATH=/content/yt-spx python run_colab_all_in_one.py --mode test-cookies


In [ ]:
# ==============================================================================
# CELL 5: INSTANT Q&A PREVIEW & QUALITY REVIEW
# Displays the newest dialectical Q&A units from the live Drive log or SQLite DB.
# Safe to run anytime without interrupting ongoing processing.
# ==============================================================================
from pathlib import Path
import json

log_file = Path('/content/drive/MyDrive/yt-spx/logs/extraction_live.log')
extractions_dir = Path('/content/drive/MyDrive/yt-spx/data/extractions')
db_file = Path('/content/drive/MyDrive/yt-spx/data/dawah_knowledge.db')

print('=' * 74)
print('  YT-SPX V2 EXTRACTION RESULTS REVIEW')
print('=' * 74)

# 1. Live stream log (updated immediately turn-by-turn)
if log_file.exists():
    content = log_file.read_text(encoding='utf-8').strip()
    blocks = [b.strip() for b in content.split('-' * 75) if b.strip()]
    print(f'\n[+] Live Stream Log: {len(blocks)} total Q&A unit(s) recorded in {log_file.name}:\n')
    for b in blocks[-3:]:
        print('-' * 74)
        print(b)
    print('-' * 74)
else:
    print('[-] Live log not created yet. (Runs automatically when Stage 2 starts).')

# 2. Detailed JSON Shards
if extractions_dir.exists():
    shards = list(extractions_dir.glob('*.qa.json'))
    print(f'\n[+] Completed Video Shards: {len(shards)} video(s)')
    if shards:
        latest = max(shards, key=lambda p: p.stat().st_mtime)
        data = json.loads(latest.read_text(encoding='utf-8'))
        units = data.get('units', []) if isinstance(data, dict) else data
        print(f'    Latest Video: {latest.stem} ({len(units)} Q&A units)')
        if units:
            sample = units[0]
            print(f'    Sample Q (AR): {sample.get("question_ar", "")}')
            print(f'    Sample Q (EN): {sample.get("question_en", "")}')
            print(f'    Sample A (AR): {sample.get("core_answer_summary_ar", "")[:140]}...')
            print(f'    Fallacy: {sample.get("fallacy_or_misconception_exposed_en", "")}')
            print(f'    URL: {sample.get("timestamp_url", "")}')

# 3. Compacted SQLite DB
if db_file.exists():
    try:
        from db.database import get_stats
        stats = get_stats(db_path=db_file)
        print(f'\n[+] Compacted SQLite DB: {stats["total_qa"]} Q&A units | {stats["stage1_completed"]} Scenarios | {stats["cleaned_hours"]}h Clean Audio')
    except Exception:
        pass
